### Tennis with MADDPG

You can run all cells or start with the "Test the trained agent" section to just load the already trained parameters and see how the agent performs.

In [ ]:
from unityagents import UnityEnvironment
import numpy as np

env = UnityEnvironment(file_name="./Tennis_Linux/Tennis.x86_64", no_graphics=True)

# get the default brain
brain_name = env.brain_names[0]
brain = env.brains[brain_name]

# reset the environment
env_info = env.reset(train_mode=True)[brain_name]

# number of agents 
num_agents = len(env_info.agents)
print('Number of agents:', num_agents)

# size of each action
action_size = brain.vector_action_space_size
print('Size of each action:', action_size)

# examine the state space 
states = env_info.vector_observations
state_size = states.shape[1]
print('There are {} agents. Each observes a state with length: {}'.format(states.shape[0], state_size))
print('The state for the first agent looks like:', states[0])


### Multiagent Deep Deterministic Policy Gradient

In [ ]:
# Import necessary libraries
from maddpg import MADDPG
from collections import deque

# Hyperparameters
ddpg_agent_params = {
    'state_size': state_size,
    'action_size': action_size,
    'hidden_size_1': 512,
    'hidden_size_2': 512,
    'gamma': 0.99,
    'tau': 1e-3,
    'lr_actor': 1e-4,
    'lr_critic': 1e-3,
    'weight_decay': 0.0001,
    'random_seed': 42,
    'theta': 0.15,
    'sigma': 0.2
}

maddpg_params = {
    'buffer_size': int(1e6),
    'batch_size': 512,
    'learn_every': 2,
    'num_training': 2,
    'random_seed': ddpg_agent_params['random_seed']
}
# Additional parameters
n_episodes = 8000 #4000
t_max = 1000

maddpg = MADDPG(num_agents, maddpg_params, ddpg_agent_params)
score_all_episodes = []
score_all_episodes_window = deque(maxlen=100)
mean_overall_score = []

# Main loop
for i_episode in range(n_episodes):
    env_info = env.reset(train_mode=True)[brain_name]     # reset the environment    

    # Initialize a random process for action exploration
    maddpg.reset()

    # Receive initial state
    states = env_info.vector_observations                  # get the current state (for each agent)
    scores = np.zeros(num_agents)                          # initialize the score (for each agent)

    for t in range(0, t_max):
        
        # Select action for each agent w.r.t current policy and exploration
        actions = maddpg.act(states, add_noise=True) # select an action (for each agent)

        env_info = env.step(actions)[brain_name]           # send all actions to tne environment

        next_states = env_info.vector_observations         # get next state (for each agent)
        rewards = env_info.rewards                         # get reward (for each agent)
        dones = env_info.local_done                        # see if episode finished

        # This adds the experience to the buffer and makes each agent learn if enough experiences 
        # are available
        maddpg.step(states, actions, rewards, next_states, dones)

        scores += env_info.rewards                         # update the score (for each agent)
        states = next_states                               # roll over states to next time step
        if np.any(dones):                                  # exit loop if episode finished
            break
    
    maddpg.set_sigma_for_all_agents()

    score_all_episodes.append(np.max(scores))                    # save most recent score
    score_all_episodes_window.append(np.max(scores))              # save most recent score
    mean_overall_score.append(np.mean(score_all_episodes_window))  # save mean score

    print(f"\rEpisode {i_episode}/{n_episodes} - Score: {np.max(scores):.2f}", end="")

    if i_episode % 10 == 0:
        print(f"\rEpisode {i_episode} - Mean Score: {np.mean(score_all_episodes_window):.2f}")

    if np.mean(score_all_episodes_window) >= 0.5:
        print(f"Environment solved in {i_episode} episodes!")
        maddpg.save_models()
        break

### Plot training results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot the scores
fig = plt.figure()
ax1 = fig.add_subplot(211)
ax1.plot(np.arange(len(score_all_episodes)), score_all_episodes)
ax1.set_ylabel('Score', fontsize=14)
ax1.set_xlabel('Episode #', fontsize=14)
ax1.tick_params(axis='both', labelsize=14)

ax2 = fig.add_subplot(212)
ax2.plot(np.arange(len(mean_overall_score)), mean_overall_score)
ax2.set_ylabel('Mean Score', fontsize=14)
ax2.set_xlabel('Episode #', fontsize=14)
ax2.tick_params(axis='both', labelsize=14)

plt.tight_layout()
plt.savefig('training_scores.png')
plt.show()

### Save trained data and parameters

In [ ]:
import json

# Save the scores and parameters to a JSON file
data = {
    'ddpg_agent_params': ddpg_agent_params,
    'maddpg_params': maddpg_params,
    'score_all_episodes': score_all_episodes,
    'mean_scores': mean_overall_score
}

with open('training_scores.json', 'w') as f:
    json.dump(data, f, indent=4)

### Test the trained agent

In [ ]:
# Test the trained agent
from maddpg import MADDPG
import json
import torch
from unityagents import UnityEnvironment
import numpy as np

env = UnityEnvironment(file_name="./Tennis_Linux/Tennis.x86_64", no_graphics=False)
brain_name = env.brain_names[0]
env_info = env.reset(train_mode=False)[brain_name]

num_agents = len(env_info.agents)# reset the environment

# Hyperparameters
# Load parameters and scores from the JSON file
with open('training_scores.json', 'r') as f:
    data = json.load(f)

maddpg = MADDPG(num_agents, data['maddpg_params'], data['ddpg_agent_params'])

for i, agent in enumerate(maddpg.agents):
    agent.actor_local.load_state_dict(torch.load(f'checkpoint_actor_{i}.pth', map_location=torch.device('cpu')))
    agent.critic_local.load_state_dict(torch.load(f'checkpoint_critic_{i}.pth', map_location=torch.device('cpu')))

#env_info = env.reset(train_mode=False)[brain_name]
states = env_info.vector_observations
scores = np.zeros(num_agents)  

t_max = 1000

for t in range(1, t_max + 1):
    actions = maddpg.act(states, add_noise=False)
    env_info = env.step(actions)[brain_name]
    next_states = env_info.vector_observations
    rewards = env_info.rewards
    dones = env_info.local_done
    scores += rewards
    states = next_states
    if any(dones):
        break

print(f"Score achieved by the trained agent: {np.max(scores)}")

# Close the environment
env.close()